# Qwen Cloud Developer Tutorial

Qwen Cloud を使った構築のハンズオンガイド — 最初の API コールから、ビジョン、画像生成、関数呼び出し、思考モードといった高度な機能まで。

**Qwen Cloud とは何ですか？**  
Qwen Cloud は、Qwen 大規模言語モデルおよびマルチモーダル AI への API アクセスを提供します。機能には、テキスト生成、ビジョン（画像 & 動画の理解）、画像生成、動画生成、音声からテキストへの変換、テキストから音声への変換、埋め込み、リランキングが含まれます — すべて OpenAI 互換および DashScope エンドポイントを通じて利用可能です。

**学ぶ内容:**

| セクション | トピック |
|---|---|
| 1 | セットアップ & インストール |
| 2 | 最初の API コール |
| 3 | テキスト生成（チャット補完） |
| 4 | ストリーミングレスポンス |
| 5 | 思考モード（推論） |
| 6 | ビジョン — 画像理解 |
| 7 | ビジョン — 動画理解 |
| 8 | 画像生成 |
| 9 | 関数呼び出し（ツール使用） |
| 10 | 構造化出力 |
| 11 | 高スループットのための非同期リクエスト |
| 12 | 次のステップ & リソース |


## 1. Setup & Installation

### 1.1 必要条件

始める前に、以下が必要です：

1. **Qwen Cloud アカウント** — [home.qwencloud.com](https://home.qwencloud.com/) でサインアップしてください
2. **API キー** — Qwen Cloud コンソールで生成してください

> **API キーは秘密にしてください！** バージョン管理にコミットしたり、公開で共有したりしないでください。

### 1.2 OpenAI Python SDK のインストール

Qwen Cloud は **OpenAI SDK と完全互換** なので、すでにおなじみの `openai` パッケージをそのまま使用できます。


In [ ]:
# Install the OpenAI Python SDK
!pip install -q openai

### 1.3 APIキーを設定する

APIキーを環境変数として設定します。以下のいずれかの方法を使用できます:

- **オプションA:** Jupyterを起動する前にターミナルでエクスポートする:  
  ```bash
  export DASHSCOPE_API_KEY="sk-your-api-key-here"
  ```

- **オプションB:** このノートブック内で直接設定する（テスト目的のみ — 絶対にこれをコミットしないでください!）:


In [ ]:
import os

# Option A: Read from environment (recommended)
# Make sure you have run: export DASHSCOPE_API_KEY="sk-your-api-key-here"

# Option B: Set directly (for quick testing only — never commit this!)
# os.environ["DASHSCOPE_API_KEY"] = "sk-your-api-key-here"

# Verify the key is set
api_key = os.getenv("DASHSCOPE_API_KEY")
if api_key:
    print(f"API key configured (starts with {api_key[:6]}...)")
else:
    print("API key not found. Please set DASHSCOPE_API_KEY.")

### 1.4 クライアントの初期化

標準的なOpenAIの使用方法との主な違いは、`base_url` です — これはQwen Cloudエンドポイントを指します。

In [ ]:
from openai import OpenAI

# Qwen Cloud uses the OpenAI-compatible endpoint
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

print("Client initialized successfully!")

### 1.5 利用可能なモデル

Qwen Cloudはさまざまなモデルを提供しています。以下はテキスト生成における主なモデルです：

| Model | Best For | Speed | Cost |
|---|---|---|---|
| `qwen3.6-max-preview` | 複雑な推論とコーディング | Slower | Higher |
| `qwen3.6-plus` | バランスの取れたパフォーマンス | Medium | Medium |
| `qwen3.6-flash` | 高速かつコスト効率が良い | Fast | Lower |

すべてのモデルは**同じAPI**を共有しています — 単に`model`パラメータを変更するだけです。品質と速度のバランスを取るには、まず`qwen3.6-plus`から始めてみてください。


---
## 2. 最初のAPI呼び出し

すべてが正常に動作していることを確認するために、簡単なリクエストを行いましょう。

In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {"role": "user", "content": "Hello! Tell me a fun fact about AI."}
    ]
)

print(completion.choices[0].message.content)

もし応答が表示された場合、おめでとうございます — Qwen Cloudに接続されています！

完全な応答構造を見てみましょう:

In [ ]:
import json

# Inspect the full API response
print(json.dumps(json.loads(completion.model_dump_json()), indent=2))

レスポンスの主なフィールド:

- `choices[0].message.content` — モデルのテキスト応答
- `choices[0].finish_reason` — モデルが停止した理由（`"stop"` = 自然な完了）
- `usage.prompt_tokens` / `usage.completion_tokens` — 課金のためのトークン数


---
## 3. テキスト生成 (Chat Completions)

Chat Completions APIはテキストを生成するための主要な方法です。リクエストは**3つのメッセージロール**で構成されています:

- **System** — アシスタントの動作と人格を設定
- **User** — あなたの入力 / プロンプト
- **Assistant** — モデルの応答

### 3.1 Systemメッセージの使用

In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer clearly and concisely."
        },
        {
            "role": "user",
            "content": "Summarize the benefits of solar energy in three bullet points."
        }
    ]
)

print(completion.choices[0].message.content)

### 3.2 Multi-Turn Conversations

ターン間のコンテキストを維持するために、各リクエストに完全なメッセージ履歴を含めてください。


In [ ]:
messages = [
    {"role": "system", "content": "You are a knowledgeable science tutor."},
    {"role": "user", "content": "What is photosynthesis?"},
]

# First turn
response1 = client.chat.completions.create(model="qwen3.6-plus", messages=messages)
assistant_reply = response1.choices[0].message.content
print("Turn 1:", assistant_reply)
print("---")

# Add assistant reply and ask a follow-up
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "Can you explain the light-dependent reactions in simpler terms?"})

# Second turn
response2 = client.chat.completions.create(model="qwen3.6-plus", messages=messages)
print("Turn 2:", response2.choices[0].message.content)

### 3.3 温度とTop-pの制御

`temperature` と `top_p` パラメータは、出力がどれだけ創造的または予測可能であるかを制御します:

| シナリオ | Temperature | Top-p |
|---|---|---|
| 創造的な文章作成 | 0.8-1.0 | 0.9-0.95 |
| コード生成 | 0.0-0.3 | 0.7-0.8 |
| 事実に基づくQ&A | 0.0-0.3 | 0.5-0.7 |
| 翻訳 | 0.0-0.3 | 0.7-0.8 |

**温度の仕組み:** 高い温度はトークンの確率分布を平坦化し、低確率のトークンが選ばれる可能性を高めます（よりランダム）。低い温度は分布を鋭くし、高確率のトークンを優先します（より予測可能）。

**Top-pの仕組み:** Top-pサンプリングは、累積確率が閾値を超える最小のトークン集合から選択します。高い `top_p` はより多くのトークンを考慮に入れます（より多様性）、低い `top_p` はより少ないトークンを考慮します（より集中）。


In [ ]:
# Creative mode — high temperature
creative = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role": "user", "content": "Write a three-sentence story about a cat and sunlight."}],
    temperature=0.9,
    top_p=0.95
)
print("Creative:")
print(creative.choices[0].message.content)
print()

# Precise mode — low temperature
precise = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role": "user", "content": "Write a three-sentence story about a cat and sunlight."}],
    temperature=0.1,
    top_p=0.7
)
print("Precise:")
print(precise.choices[0].message.content)

## 4. ストリーミングレスポンス

より良いユーザーエクスペリエンスのために、トークンごとにレスポンスをストリーミングすることができます。これは特に長い出力やチャットインターフェースにおいて有用です。


In [ ]:
stream = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain quantum computing in simple terms."}
    ],
    stream=True  # Enable streaming
)

print("Streaming response:")
full_response = ""
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        full_response += content
        print(content, end="", flush=True)

print("\n\n--- Stream complete ---")

## 5. Thinking Mode (Reasoning)

Thinking mode はモデルの内部推論プロセスを公開します。これは、複雑な数学、論理、およびコーディングタスクにおいて非常に価値があります。有効化すると、モデルは以下を返します：

- **Phase 1: Thinking** — ステップバイステップの推論を示す `reasoning_content`
- **Phase 2: Answer** — 最終的な応答を含む `content`

`enable_thinking` を `True` に設定することで有効化できます。


In [ ]:
stream = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role": "user", "content": "If 3x + 7 = 22, what is x?"}],
    extra_body={"enable_thinking": True},  # Enable thinking mode
    stream=True
)

thinking_content = ""
answer_content = ""

print("Thinking process:")
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    
    # Phase 1: Reasoning / thinking
    if hasattr(delta, "reasoning_content") and delta.reasoning_content:
        thinking_content += delta.reasoning_content
        print(delta.reasoning_content, end="", flush=True)
    
    # Phase 2: Final answer
    if delta.content:
        if not answer_content:  # Print header on first answer token
            print("\n\nAnswer:")
        answer_content += delta.content
        print(delta.content, end="", flush=True)

print()

### 5.1 `thinking_budget`で思考の深さを制御する

モデルが推論に費やすトークン数を制限できます:

In [ ]:
# Limit thinking to 500 tokens (faster, less detailed reasoning)
stream = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role": "user", "content": "What is the integral of x^2 from 0 to 5?"}],
    extra_body={
        "enable_thinking": True,
        "thinking_budget": 500  # Max tokens for reasoning
    },
    stream=True
)

for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if hasattr(delta, "reasoning_content") and delta.reasoning_content:
        print(delta.reasoning_content, end="", flush=True)
    if delta.content:
        print(delta.content, end="", flush=True)

print()

## 6. Vision — Image Understanding

Qwen Cloudのビジョンモデルは、画像を分析して質問に答えたり、テキストを抽出（OCR）したり、内容を説明したり、視覚的な問題を解決したり、スクリーンショットからコードを生成したりすることができます。

### 6.1 Analyze a Single Image from URL


In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"
                    }
                },
                {
                    "type": "text",
                    "text": "Describe what you see in this image."
                }
            ]
        }
    ]
)

print(completion.choices[0].message.content)

### 6.2 複数画像の分析

比較や統合分析のために、1回のリクエストで複数の画像を渡すことができます。


In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"
                    }
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://dashscope.oss-cn-beijing.aliyuncs.com/images/tiger.png"
                    }
                },
                {
                    "type": "text",
                    "text": "Compare these two images. What do they depict?"
                }
            ]
        }
    ]
)

print(completion.choices[0].message.content)

### 6.3 ローカル画像を分析する (Base64)

ローカルファイルの場合、Base64としてエンコードし、インラインで渡します。

In [ ]:
import base64

def encode_image(image_path: str) -> str:
    """Convert a local image file to a Base64-encoded string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Example usage (uncomment and set your image path):
# base64_image = encode_image("/path/to/your/image.png")
#
# completion = client.chat.completions.create(
#     model="qwen3.6-plus",
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "image_url",
#                     "image_url": {"url": f"data:image/png;base64,{base64_image}"}
#                 },
#                 {"type": "text", "text": "Describe what you see in this image."}
#             ]
#         }
#     ]
# )
# print(completion.choices[0].message.content)

print("encode_image() helper function is ready to use.")
print("Uncomment the example above and set your image path to try it.")

## 7. Vision — Video Understanding

Qwen Cloudは動画コンテンツも分析できます — 何が起こっているかを要約し、イベントを特定し、説明を生成します。


In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241115/cqqkru/1.mp4"
                    },
                    "fps": 2  # Extract 2 frames per second
                },
                {
                    "type": "text",
                    "text": "Summarize what happens in this video."
                }
            ]
        }
    ]
)

print(completion.choices[0].message.content)

> **Tip:** `fps` パラメーターは、ビデオから抽出される1秒あたりのフレーム数を制御します。`fps` の値を低くすると使用されるトークンが少なくなりますが、動きの速い詳細を見逃す可能性があります。

## 8. 画像生成

Qwen Cloudは、テキストプロンプトから画像を生成できる強力な画像生成モデル（Wanシリーズ）を提供しています。

画像生成は、非同期タスクポーリングを使用した**DashScopeネイティブAPI**を利用します。以下が完全なワークフローです：

1. **非同期生成タスクを送信**
2. タスクのステータスを**ポーリング**し、完了を待つ
3. 生成された画像のURLを**取得**

### 利用可能な画像モデル

| Model | Description |
|---|---|
| `wan2.7-image-pro` | 最高品質、最大4096x4096をサポート |
| `wan2.7-image` | 高品質、最大2048x2048をサポート |
| `wan2.6-image` | 前世代モデル、最大1280x1280 |
| `qwen-image-plus` | 固定プリセット、デフォルト1664x928 |


In [ ]:
import requests
import time

API_KEY = os.getenv("DASHSCOPE_API_KEY")
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-DashScope-Async": "enable"  # Enable async mode
}

# Step 1: Submit the image generation task
payload = {
    "model": "wan2.7-image",
    "input": {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": "A serene mountain lake at sunrise, with mist rolling over calm water, "
                                "pine trees reflected in the surface, photorealistic style"
                    }
                ]
            }
        ]
    },
    "parameters": {
        "size": "1024x1024",
        "n": 1
    }
}

response = requests.post(
    "https://dashscope-intl.aliyuncs.com/api/v1/services/aigc/image-generation/generation",
    headers=HEADERS,
    json=payload
)

result = response.json()
task_id = result["output"]["task_id"]
print(f"Task submitted! Task ID: {task_id}")
print(f"Status: {result['output']['task_status']}")

In [ ]:
# Step 2: Poll the task until it completes
poll_headers = {
    "Authorization": f"Bearer {API_KEY}"
}

while True:
    status_response = requests.get(
        f"https://dashscope-intl.aliyuncs.com/api/v1/tasks/{task_id}",
        headers=poll_headers
    )
    status_result = status_response.json()
    task_status = status_result["output"]["task_status"]
    
    print(f"Status: {task_status}")
    
    if task_status in ["SUCCEEDED", "FAILED"]:
        break
    
    time.sleep(3)  # Wait 3 seconds before polling again

# Step 3: Get the generated image URL
if task_status == "SUCCEEDED":
    image_url = status_result["output"]["results"][0]["url"]
    print(f"\nImage generated successfully!")
    print(f"URL: {image_url}")
else:
    print(f"\nTask failed: {status_result}")

In [ ]:
# Display the generated image inline (optional)
from IPython.display import Image, display

if task_status == "SUCCEEDED":
    display(Image(url=image_url, width=512))
else:
    print("No image to display — the task did not succeed.")

## 9. Function Calling (Tool Use)

Function callingは、モデルがAPI、データベース、またはカスタム関数などの外部ツールを使用して、自力では解決できない質問に答えることを可能にします。

**仕組み:**
1. ユーザーの質問 **+ 利用可能なツールのリスト** をモデルに送信します
2. モデルがどのツールを呼び出すかを決定し、ツール名 + パラメータを返します
3. アプリケーションがツールを実行し、結果を取得します
4. ツールの結果をモデルに返します
5. モデルが最終的な自然言語の応答を生成します

### 9.1 Define Your Tools


In [ ]:
import json

# Define the tools the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a specific city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, e.g. Singapore or New York"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

# Simulate the tool (in production, this would call a real weather API)
def get_current_weather(location: str) -> str:
    """Simulated weather lookup."""
    weather_data = {
        "Singapore": "Partly cloudy, 31C, humidity 78%",
        "New York": "Sunny, 22C, humidity 45%",
        "London": "Overcast, 15C, humidity 82%",
    }
    return weather_data.get(location, f"Weather data unavailable for {location}")

print("Tools and simulated function defined.")

### 9.2 完全な関数呼び出しワークフロー

In [ ]:
messages = [
    {"role": "user", "content": "What is the weather like in Singapore today?"}
]

# Step 1: First model call — the model decides to call a tool
response = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
print("Step 1 — Model wants to call a tool:")

if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    print(f"   Tool: {function_name}")
    print(f"   Args: {function_args}")
    
    # Step 2: Execute the tool
    tool_result = get_current_weather(**function_args)
    print(f"\nStep 2 — Tool result: {tool_result}")
    
    # Step 3: Send the tool result back to the model
    messages.append(assistant_message.model_dump())  # Add assistant tool call message
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": tool_result
    })
    
    # Step 4: Second model call — the model generates the final answer
    final_response = client.chat.completions.create(
        model="qwen3.6-plus",
        messages=messages,
        tools=tools
    )
    
    print(f"\nStep 3 — Final response:")
    print(final_response.choices[0].message.content)
else:
    print("   Model responded directly (no tool call needed):")
    print(f"   {assistant_message.content}")

### 9.3 Parallel Tool Calls

ユーザーが複数の独立した事柄について質問した場合、モデルは複数のツールを同時に呼び出すことができます。


In [ ]:
messages = [
    {"role": "user", "content": "What is the weather in Singapore and London?"}
]

response = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=messages,
    tools=tools,
    parallel_tool_calls=True  # Allow multiple tool calls in one response
)

assistant_message = response.choices[0].message

if assistant_message.tool_calls:
    print(f"Model requested {len(assistant_message.tool_calls)} parallel tool calls:\n")
    
    messages.append(assistant_message.model_dump())
    
    # Execute all tool calls
    for tool_call in assistant_message.tool_calls:
        args = json.loads(tool_call.function.arguments)
        result = get_current_weather(**args)
        print(f"  Tool: {tool_call.function.name}({args}) -> {result}")
        
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
    
    # Get final response
    final = client.chat.completions.create(
        model="qwen3.6-plus",
        messages=messages,
        tools=tools
    )
    print(f"\nFinal answer:\n{final.choices[0].message.content}")

### 9.4 強制的なツール選択

`tool_choice` を使用して、特定のツールを呼び出す（または呼び出さない）ようにモデルを強制できます:

```python
# 特定のツールを強制的に使用
tool_choice={"type": "function", "function": {"name": "get_current_weather"}}

# すべてのツールをブロック（テキストのみの応答を強制）
tool_choice="none"
```

### 9.5 関数呼び出しのベストプラクティス

- **ツール選択の精度をテスト** してから本番環境に移行する
- **ツールの説明を明確に保つ** — モデルは説明を使用して各ツールを呼び出すタイミングを判断する
- **ツールの数を制限する** — 候補セットが小さいほど精度が向上する
- **書き込み操作に対して人間の確認を追加する** （例: メール送信、購入処理）
- **タイムアウトとフォールバックを設定する** — ツールが失敗する可能性があるため、エラー処理を適切に行う
- **注意:** ツールの説明は入力トークンとしてカウントされ、プロンプトの一部として課金されます


---
## 10. 構造化出力

モデルに特定のスキーマに準拠したJSONを返すよう指示できます。これは、非構造化テキストから構造化データを抽出する際に役立ちます。

### 10.1 JSONモード（シンプル）

In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "system",
            "content": "You extract contact information from text. Return a JSON object with keys: name, email, phone, company."
        },
        {
            "role": "user",
            "content": "Hi, I am Sarah Chen from TechCorp. You can reach me at sarah.chen@techcorp.com or call 555-0142."
        }
    ],
    response_format={"type": "json_object"}
)

result = json.loads(completion.choices[0].message.content)
print(json.dumps(result, indent=2))

### 10.2 JSON Schemaモード（厳密）

より厳密な制御のために、モデルが準拠しなければならないJSON Schemaを提供します:

In [ ]:
completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[
        {
            "role": "system",
            "content": "You are a recipe analyzer. Extract the recipe details from the user text."
        },
        {
            "role": "user",
            "content": "To make a classic margherita pizza, you need pizza dough, 200g mozzarella, 150ml tomato sauce, fresh basil leaves, olive oil, and a pinch of salt. Preheat oven to 250C, spread sauce on dough, add cheese, bake for 10-12 minutes, then top with basil."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "recipe",
            "schema": {
                "type": "object",
                "properties": {
                    "dish_name": {"type": "string"},
                    "ingredients": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "steps": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "cooking_time_minutes": {"type": "integer"},
                    "temperature_celsius": {"type": "integer"}
                },
                "required": ["dish_name", "ingredients", "steps", "cooking_time_minutes", "temperature_celsius"]
            }
        }
    }
)

recipe = json.loads(completion.choices[0].message.content)
print(json.dumps(recipe, indent=2))

## 11. 高スループットのための非同期リクエスト

高い同時実行性のワークロードの場合、`AsyncOpenAI` を使用して複数のリクエストを並行して送信します。


In [ ]:
import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

async def ask_question(question: str) -> str:
    """Send a single async request."""
    print(f"Sent: {question}")
    response = await async_client.chat.completions.create(
        model="qwen3.6-plus",
        messages=[{"role": "user", "content": question}]
    )
    answer = response.choices[0].message.content
    print(f"Received answer for: {question[:50]}...")
    return answer

async def main():
    questions = [
        "Summarize the benefits of solar energy in three bullet points.",
        "Write a subject line for a product launch email.",
        "Translate 'Welcome to our platform' into Spanish."
    ]
    
    # Send all questions concurrently
    results = await asyncio.gather(*[ask_question(q) for q in questions])
    
    print("\n" + "=" * 60)
    for q, a in zip(questions, results):
        print(f"\nQ: {q}")
        print(f"A: {a}")

# In Jupyter notebooks, use await directly
await main()

> **Note:** Jupyterノートブックでは、Jupyterが既に非同期イベントループを実行しているため、`await main()` を直接使用できます。通常のPythonスクリプトでは、`asyncio.run(main())` を使用してください。

## 12. 次のステップとリソース

おめでとうございます！Qwen Cloudの主要な機能を探索しました。以下は、これまでに学んだ内容と次に進むべき方向の概要です。

### 学んだ内容

| セクション | 機能 | 主なAPI |
|---|---|---|
| 2-3 | テキスト生成 | `chat.completions.create()` |
| 4 | ストリーミング | `stream=True` |
| 5 | 思考 / 推論 | `enable_thinking=True` |
| 6-7 | ビジョン（画像 & 動画） | `image_url` / `video_url` コンテンツタイプ |
| 8 | 画像生成 | DashScope 非同期API |
| 9 | 関数呼び出し | `tools` パラメータ |
| 10 | 構造化出力 | `response_format` |
| 11 | 非同期リクエスト | `AsyncOpenAI` |

### APIエンドポイントリファレンス

| APIスタイル | ベースURL |
|---|---|
| OpenAI互換 — Chat Completions | `https://dashscope-intl.aliyuncs.com/compatible-mode/v1` |
| OpenAI互換 — Responses API | `https://dashscope-intl.aliyuncs.com/api/v2/apps/protocols/compatible-mode/v1` |
| DashScope — テキスト生成 | `https://dashscope-intl.aliyuncs.com/api/v1/services/aigc/text-generation/generation` |
| DashScope — マルチモーダル生成 | `https://dashscope-intl.aliyuncs.com/api/v1/services/aigc/multimodal-generation/generation` |
| DashScope — 画像生成 | `https://dashscope-intl.aliyuncs.com/api/v1/services/aigc/image-generation/generation` |

### さらに探求する

- [Model Gallery](https://www.qwencloud.com/models) — 利用可能なすべてのモデルを閲覧
- [Text Generation Guide](https://docs.qwencloud.com/developer-guides/text-generation/quickstart) — テキスト生成の詳細ガイド
- [Vision Guide](https://docs.qwencloud.com/developer-guides/multimodal/vision) — 高度な画像および動画分析
- [Image Generation](https://docs.qwencloud.com/developer-guides/image-generation/text-to-image) — 画像編集、Wanモデル
- [Video Generation](https://docs.qwencloud.com/developer-guides/video-generation/text-to-video) — テキストから動画、画像から動画
- [Speech (TTS)](https://docs.qwencloud.com/developer-guides/speech/tts-models) — 音声クローンを用いたテキスト読み上げ
- [Function Calling](https://docs.qwencloud.com/developer-guides/text-generation/function-calling) — エージェントアプリの構築
- [Structured Output](https://docs.qwencloud.com/developer-guides/text-generation/structured-output) — JSONスキーマの適用
- [Pricing](https://docs.qwencloud.com/developer-guides/getting-started/pricing) — 料金詳細
- [Free Quota](https://docs.qwencloud.com/resources/free-quota) — 利用可能な無料クレジット
- [API Reference](https://docs.qwencloud.com/api-reference/preparation/api-key) — 完全なAPIドキュメント


---

*このチュートリアルは、[Qwen Cloud Developer Guide](https://docs.qwencloud.com/developer-guides/getting-started/introduction) に基づいて作成されました。最新の更新情報や追加機能については、公式ドキュメントをご覧ください。*